# Synthetic Data Generator Pipeline
### Step 5 : Build Send Queue Stage


In [ ]:
from utils.postgres_util import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.send_queue_stage_writer import (
    build_sensor_messages_send_queue,
    validate_sensor_messages_send_queue,
)

In [ ]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")


IF_EXISTS_FLAG = str("replace")
QUEUE_STATUS_DEFAULT = str("pending")
CHUNK_SIZE = int(50000)

SEND_QUEUE_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_stage")
SEND_QUEUE_DESTINATION_TABLE_NAME = str("synthetic_sensor_messages_send_queue")

QUEUE_OWNER_ROLE = str("kafka_producer")
APPLY_OWNER_AND_GRANTS = bool(True)

---

In [2]:


engine = get_engine_from_env()


---

In [ ]:
send_queue_table_name = build_sensor_messages_send_queue(
    engine=engine,
    schema=SCHEMA,
    source_table=SEND_QUEUE_SOURCE_TABLE_NAME,
    target_table=SEND_QUEUE_DESTINATION_TABLE_NAME,
    if_exists=IF_EXISTS_FLAG,
    queue_status_default=QUEUE_STATUS_DEFAULT,
    chunk_size=CHUNK_SIZE,
    queue_owner_role=QUEUE_OWNER_ROLE,
    apply_owner_and_grants=APPLY_OWNER_AND_GRANTS,
)

print("Built queue table:", send_queue_table_name)

[chunk] 1 | source rows 0 to 49,999
[send-queue] wrote chunk 1 with 50,000 rows
[chunk] 2 | source rows 50,000 to 99,999
[send-queue] wrote chunk 2 with 50,000 rows
[chunk] 3 | source rows 100,000 to 149,999
[send-queue] wrote chunk 3 with 50,000 rows
[chunk] 4 | source rows 150,000 to 199,999
[send-queue] wrote chunk 4 with 50,000 rows
[chunk] 5 | source rows 200,000 to 249,999
[send-queue] wrote chunk 5 with 50,000 rows
[chunk] 6 | source rows 250,000 to 299,999
[send-queue] wrote chunk 6 with 50,000 rows
[chunk] 7 | source rows 300,000 to 349,999
[send-queue] wrote chunk 7 with 50,000 rows
[chunk] 8 | source rows 350,000 to 399,999
[send-queue] wrote chunk 8 with 50,000 rows
[chunk] 9 | source rows 400,000 to 449,999
[send-queue] wrote chunk 9 with 50,000 rows
[chunk] 10 | source rows 450,000 to 499,999
[send-queue] wrote chunk 10 with 50,000 rows
[chunk] 11 | source rows 500,000 to 549,999
[send-queue] wrote chunk 11 with 50,000 rows
[chunk] 12 | source rows 550,000 to 599,999
[sen

---

In [7]:
validation_dataframe = validate_sensor_messages_send_queue(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_DESTINATION_TABLE_NAME,
)

display(validation_dataframe)

,row_count,distinct_observation_count,distinct_sensor_name_count,min_observation_timestamp,max_observation_timestamp,pending_count,unsent_count
0,3744000,72000,52,2026-03-19 08:00:00+00:00,2026-05-08 07:59:00+00:00,3744000,3744000


---

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        message_key,
        observation_index,
        observation_timestamp,
        message_sequence_index,
        sensor_name,
        sensor_index,
        sensor_value,
        queue_status,
        claim_token,
        claimed_at,
        producer_topic,
        producer_worker_id,
        queued_at,
        producer_sent_at,
        producer_ack_at,
        producer_delivery_status,
        producer_delivery_error
    FROM {SCHEMA}.{SEND_QUEUE_DESTINATION_TABLE_NAME}
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 104
    """
)

display(sample_dataframe)

---

In [11]:
pending_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT *
    FROM capstone.synthetic_sensor_messages_send_queue
    WHERE queue_status = 'pending'
      AND producer_sent_at IS NULL
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 500
    """
)

display(pending_dataframe.head(100))

,dataset_id,run_id,asset_id,message_key,generated_row_id,observation_index,observation_timestamp,message_sequence_index,batch_id,row_in_batch,...,sensor_index,sensor_value,is_telemetry_event,telemetry_event_type,producer_send_attempt,queue_status,queued_at,producer_sent_at,producer_delivery_status,producer_delivery_error
0,pump_synthetic_v1,premelt_run_001,pump_asset_001,pump_asset_001|1|41,premelt_run_001_obs_000000000001,1,2026-03-19 08:00:00+00:00,0,1,0,...,41,44.118690,False,None,1,pending,2026-04-04 04:53:38.620967+00:00,None,None,None
1,pump_synthetic_v1,premelt_run_001,pump_asset_001,pump_asset_001|1|49,premelt_run_001_obs_000000000001,1,2026-03-19 08:00:00+00:00,1,1,0,...,49,59.773464,False,None,1,pending,2026-04-04 04:53:38.620967+00:00,None,None,None
2,pump_synthetic_v1,premelt_run_001,pump_asset_001,pump_asset_001|1|46,premelt_run_001_obs_000000000001,1,2026-03-19 08:00:00+00:00,2,1,0,...,46,48.795368,False,None,1,pending,2026-04-04 04:53:38.620967+00:00,None,None,None
3,pump_synthetic_v1,premelt_run_001,pump_asset_001,pump_asset_001|1|32,premelt_run_001_obs_000000000001,1,2026-03-19 08:00:00+00:00,3,1,0,...,32,571.445862,False,None,1,pending,2026-04-04 04:53:38.620967+00:00,None,None,None
4,pump_synthetic_v1,premelt_run_001,pump_asset_001,pump_asset_001|1|18,premelt_run_001_obs_000000000001,1,2026-03-19 08:00:00+00:00,4,1,0,...,18,3.253774,False,None,1,pending,2026-04-04 04:53:38.620967+00:00,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,pump_synthetic_v1,premelt_run_001,pump_asset_001,pump_asset_001|2|42,premelt_run_001_obs_000000000002,2,2026-03-19 08:01:00+00:00,43,1,1,...,42,34.694111,False,None,1,pending,2026-04-04 04:53:38.620967+00:00,None,None,None
96,pump_synthetic_v1,premelt_run_001,pump_asset_001,pump_asset_001|2|31,premelt_run_001_obs_000000000002,2,2026-03-19 08:01:00+00:00,44,1,1,...,31,618.769104,False,None,1,pending,2026-04-04 04:53:38.620967+00:00,None,None,None
97,pump_synthetic_v1,premelt_run_001,pump_asset_001,pump_asset_001|2|32,premelt_run_001_obs_000000000002,2,2026-03-19 08:01:00+00:00,45,1,1,...,32,675.997009,False,None,1,pending,2026-04-04 04:53:38.620967+00:00,None,None,None
98,pump_synthetic_v1,premelt_run_001,pump_asset_001,pump_asset_001|2|34,premelt_run_001_obs_000000000002,2,2026-03-19 08:01:00+00:00,46,1,1,...,34,281.032654,False,None,1,pending,2026-04-04 04:53:38.620967+00:00,None,None,None


---

In [ ]:
ownership_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        schemaname,
        tablename,
        tableowner
    FROM pg_tables
    WHERE schemaname = :schema_name
      AND tablename = :table_name
    """,
    params={
        "schema_name": SCHEMA,
        "table_name": SEND_QUEUE_DESTINATION_TABLE_NAME,
    },
)

display(ownership_dataframe)

---

In [12]:
# Dispose Engine
engine.dispose()